In [1]:
import jax
import jax.numpy as jnp

# --- 1. Parameter Initialization ---
def init_mlp_params(key, layer_sizes):
    # layer_sizes will be a list: [784, 128, 10]
    params = []

    # We need to loop through the layers to create weights and biases
    for i in range(len(layer_sizes) - 1):
        in_size = layer_sizes[i]
        out_size = layer_sizes[i + 1]

        # TODO: 1. Split the JAX PRNG key into 'key' and 'subkey'
        key,subkey=jax.random.split(key)

        # TODO: 2. Generate weights 'w' of shape (in_size, out_size).
        # Tip: Multiply by 0.01 to keep initial weights small!
        w=jax.random.normal(subkey,shape=(in_size, out_size))*0.01

        # TODO: 3. Generate biases 'b' of shape (out_size,) initialized to zeros.

        b=jnp.zeros(out_size)

        params.append((w, b))

        # Append this layer's parameters to our list as a tuple
        # params.append((w, b))


    return params

# --- 2. The Forward Pass ---
def predict_mlp(params, x):
    # x is a single flattened image of shape (784,)

    # params is our list of tuples: [(w1, b1), (w2, b2)]
    # We can unpack them directly!
    w1, b1 = params[0] # Hidden Layer
    w2, b2 = params[1] # Output Layer

    # TODO: 1. Compute the hidden layer linear step: dot product of x and w1, plus b1
    hidden_z = jnp.dot(x,w1) + b1

    # TODO: 2. Apply the ReLU non-linearity.
    # Tip: Use jax.nn.relu()
    hidden_a = jax.nn.relu(hidden_z)

    # TODO: 3. Compute the output layer linear step: dot product of hidden_a and w2, plus b2
    logits = jnp.dot(hidden_a,w2) + b2

    # TODO: 4. Return the probabilities using jax.nn.softmax(logits)
    return jax.nn.softmax(logits)


In [2]:
# --- We create the batched forward pass first ---
batched_predict = jax.vmap(predict_mlp, in_axes=(None, 0))

# --- 3. The Loss Function ---
def cross_entropy_loss(params, X_batch, y_batch):
    # y_batch are ONE-HOT encoded labels, shape (32, 10)

    # 1. Get predictions for the whole batch
    predictions = batched_predict(params, X_batch)

    # 2. To avoid log(0) exploding to infinity, we add a tiny number (1e-7)
    # TODO: Calculate the cross-entropy math: -
    loss = -jnp.mean(jnp.sum(y_batch*jnp.log(predictions + 1e-7),axis=1))

    return loss

# --- 4. The Update Engine ---
@jax.jit
def update_mlp_step(params, X_batch, y_batch, learning_rate):
    # TODO: 1. Create the gradient function using jax.grad on cross_entropy_loss.
    # Note: params is the 0th argument.
    calculate_gradient = jax.grad(cross_entropy_loss,0)
    # TODO: 2. Get the gradients by calling the new function
    grads = calculate_gradient(params, X_batch, y_batch)

    # TODO: 3. Update all layers.

    # Because 'params' is a list of tuples: [(w1,b1), (w2,b2)]
    # and 'grads' is exactly the same shape, we can use a python list comprehension!

    new_params = [
        (w - learning_rate * dw, b - learning_rate * db)
        for (w, b), (dw, db) in zip(params, grads)
    ]

    return new_params

In [3]:
import time

if __name__ == "__main__":
    print("Initializing Neural Network Engine...")

    # 784 input pixels (28x28 image), 128 hidden neurons, 10 output classes (digits 0-9)
    layer_sizes = [784, 128, 10]
    key = jax.random.PRNGKey(42)

    # Build the initial brain
    params = init_mlp_params(key, layer_sizes)

    # Generate Fake MNIST Data (100 images)
    # X is 100 images, each with 784 pixels
    X_train = jax.random.normal(key, (100, 784))

    # y is 100 labels, one-hot encoded (e.g., [0, 0, 1, 0, ...])
    # We will just generate random probabilities and use argmax to create pure 1s and 0s
    random_labels = jax.random.uniform(key, (100, 10))
    y_train = jnp.eye(10)[jnp.argmax(random_labels, axis=1)]

    epochs = 500
    learning_rate = 0.5

    print("Starting Training Loop...")
    start_time = time.time()

    for epoch in range(epochs):
        # Pass the parameters through the JIT-compiled factory
        params = update_mlp_step(params, X_train, y_train, learning_rate)

        if epoch % 50 == 0:
            loss = cross_entropy_loss(params, X_train, y_train)
            print(f"Epoch {epoch:3d} | Cross-Entropy Loss: {loss:.4f}")

    print(f"Training complete in {time.time() - start_time:.2f} seconds.")

Initializing Neural Network Engine...
Starting Training Loop...
Epoch   0 | Cross-Entropy Loss: 2.2518
Epoch  50 | Cross-Entropy Loss: 0.0053
Epoch 100 | Cross-Entropy Loss: 0.0021
Epoch 150 | Cross-Entropy Loss: 0.0013
Epoch 200 | Cross-Entropy Loss: 0.0009
Epoch 250 | Cross-Entropy Loss: 0.0007
Epoch 300 | Cross-Entropy Loss: 0.0006
Epoch 350 | Cross-Entropy Loss: 0.0005
Epoch 400 | Cross-Entropy Loss: 0.0004
Epoch 450 | Cross-Entropy Loss: 0.0003
Training complete in 0.52 seconds.
